 Imports & path

In [2]:
from pathlib import Path
import sys
import json

_src = Path.cwd() / "notebooks" / "src"
if not _src.is_dir():
    _src = Path.cwd().parent / "notebooks" / "src"
sys.path.insert(0, str(_src))
from paths import PROJECT_ROOT

from openai_analyzer import analyze_complaint
from pipeline import full_analysis


One manual test

In [3]:
text = """
I disputed a charge on my credit report but the company keeps reporting 
incorrect late payments from two years ago. I sent proof twice and they 
closed my case without fixing it.
"""

# Option A: full pipeline
result = full_analysis(text)
print(json.dumps(result, indent=2))

{
  "cfpb_issue_label": "Problem with a company's investigation into an existing problem",
  "quality_category": "Process delay issue",
  "manufacturing_demo_category": "Process control / service delay",
  "predicted_category": "Process delay issue",
  "urgency": "Medium",
  "likely_root_cause": "The company may have inadequate processes for handling disputes or insufficient communication regarding case status. Additionally, there may be a lack of follow-up on submitted evidence, leading to unresolved issues.",
  "containment_action": "Review the disputed case immediately and ensure all submitted evidence is properly documented and considered. Reach out to the customer to acknowledge the oversight and provide an update.",
  "corrective_action": "Implement a more robust tracking system for dispute cases to ensure timely follow-up and resolution. Provide additional training for staff on handling disputes and maintaining clear communication with customers.",
  "customer_response": "Dear C

 Another category (billing)

In [4]:
text = "they charged my account twice and will not fix my statement"
print(json.dumps(full_analysis(text), indent=2))

{
  "cfpb_issue_label": "Problem with a purchase shown on your statement",
  "quality_category": "Unexpected charge or cost issue",
  "manufacturing_demo_category": "Billing or specification mismatch",
  "predicted_category": "Unexpected charge or cost issue",
  "urgency": "Medium",
  "likely_root_cause": "The issue may stem from a system error that resulted in a duplicate charge on the customer's account. Additionally, there may be a lack of effective communication or resolution processes in place for addressing billing discrepancies.",
  "containment_action": "Investigate the charge immediately and issue a refund for the duplicate transaction if confirmed. Ensure the customer is informed of the steps being taken.",
  "corrective_action": "Review and enhance the billing system to prevent duplicate charges and improve the customer service process for handling such complaints more efficiently.",
  "customer_response": "Dear Customer, thank you for bringing this matter to our attention. 

Safety / high urgency example

In [5]:
text = """
They threatened to sue me and garnish my wages for a debt I already paid. 
I have bank statements but they refuse to review them.
"""
print(json.dumps(full_analysis(text), indent=2))

{
  "cfpb_issue_label": "Took or threatened to take negative or legal action",
  "quality_category": "Safety or high-risk issue",
  "manufacturing_demo_category": "Urgent quality / safety risk",
  "predicted_category": "Safety or high-risk issue",
  "urgency": "High",
  "likely_root_cause": "The complaint indicates a potential misunderstanding or miscommunication regarding the debt status. The refusal to review bank statements may suggest a lack of proper procedures in handling disputes over payments.",
  "containment_action": "Immediately review the customer's account and the associated payment records to verify the claim and halt any further collection actions until the matter is resolved.",
  "corrective_action": "Implement a more robust process for reviewing payment disputes, including a clear protocol for examining supporting documentation provided by customers.",
  "customer_response": "We appreciate you bringing this matter to our attention. We take your concerns seriously and a

 10-row batch to CSV (for Step 9 evaluation)

In [6]:
import pandas as pd

df = pd.read_csv(PROJECT_ROOT / "data/processed/complaints_clean.csv").sample(10, random_state=42)
rows = []
for _, row in df.iterrows():
  try:
    out = full_analysis(row["complaint_text"])
    rows.append(out)
  except Exception as e:
    rows.append({"error": str(e), "complaint_text": row["complaint_text"][:100]})

pd.DataFrame(rows).to_csv(PROJECT_ROOT / "outputs/openai_sample_10.csv", index=False)

## Step 7 — OpenAI assistant summary

### What we built

After the baseline classifier (Step 5) predicts a CFPB `issue_label` and the taxonomy layer (Step 6) maps it to a **general quality category**, OpenAI generates structured guidance for quality teams and a draft customer reply.

**End-to-end flow:**



### Modules

| File | Role |
|------|------|
| `notebooks/src/openai_analyzer.py` | `analyze_complaint()` — OpenAI call, JSON output, light validation |
| `notebooks/src/pipeline.py` | `full_analysis()` — ML + taxonomy + OpenAI in one function |
| `.env` | `OPENAI_API_KEY` (not committed) |

### Output JSON (Step 7.2 contract)

| Field | Source |
|-------|--------|
| `predicted_category` | Set from taxonomy (`quality_category`); enforced to match input |
| `urgency` | OpenAI — `Low` / `Medium` / `High` / `Critical` |
| `likely_root_cause` | OpenAI — 1–3 sentences |
| `containment_action` | OpenAI — immediate steps |
| `corrective_action` | OpenAI — longer-term prevention |
| `customer_response` | OpenAI — draft reply (4–6 sentences) |

`full_analysis()` also returns: `cfpb_issue_label`, `quality_category`, `manufacturing_demo_category`.

### API settings

- **Model:** `gpt-4o-mini`
- **Format:** `response_format={"type": "json_object"}`
- **Temperature:** 0.3

### Manual tests (smoke checks)

| Example | ML issue (summary) | Quality category | Urgency |
|---------|------------------|------------------|---------|
| Credit dispute / case closed without fix | Investigation problem | Process delay issue | Medium |
| Double charge on statement | Purchase on statement | Unexpected charge or cost issue | Medium |
| Legal threat / debt already paid | Negative or legal action | Safety or high-risk issue | High |

Outputs were coherent: containment and corrective actions aligned with the complaint; `predicted_category` matched the taxonomy label.

### Batch sample for Step 9

- **File:** `outputs/openai_sample_10.csv`
- **Rows:** 10 complaints (`random_state=42`) from `complaints_clean.csv`
- **Purpose:** manual quality scoring in Step 9 (usefulness rubric 1–5)

### Status

**Main Step 7 complete.** Ready for **Step 8** (Streamlit: paste text → `full_analysis()` → display results).

---

### Limitations

1. **Not manufacturing-trained** — The assistant reasons over **public financial complaint text**. Actions are framed in general quality language; they are not verified against real plant procedures or standards.

2. **ML error propagates** — If the baseline model misclassifies the CFPB issue, the wrong `quality_category` is sent to OpenAI and the whole analysis may be misaligned.

3. **No automated quality score for generated text** — Accuracy, precision, and F1 apply only to the **classifier** (Step 5). Root cause, actions, and customer replies must be judged **manually** (Step 9).

4. **LLM hallucination risk** — The model may suggest plausible but incorrect root causes or actions. **Do not send** `customer_response` to real customers without human review.

5. **API dependency** — Requires `OPENAI_API_KEY`, network access, and incurs cost/latency per call. The 10-row batch is for evaluation only, not production scale.

6. **Generic / templated tone** — Responses can sound similar across complaints. Tuning prompts, few-shot examples, or a stronger model may help but was out of scope for v1.

7. **Privacy** — Do not paste real PII into demos. CFPB narratives are redacted (`xxxx`), but production use would need data-handling policies.

8. **Urgency is model-judged** — `urgency` is not rule-based. Edge cases may need business rules (e.g. always `Critical` for certain keywords) in a future version.